# 📊 モデルの動作を理解する - 推論パイプライン解析
**LLM が回答を生成する仕組みの詳細**

---

## 📚 目次
1. [パイプライン全体をステップで追う](#1-パイプライン全体をステップで追う)
2. [ハイパーパラメータの効果比較](#2-ハイパーパラメータの効果比較)
3. [バッチ推論と最適化](#3-バッチ推論と最適化)
4. [トラブルシューティング](#4-トラブルシューティング)

## 1. パイプライン全体をステップで追う

**入力テキスト → トークン化 → 埋め込み → Transformer → 確率 → 単語選択**

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# GPT-2 をロード（軽量なので手元で動く）
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
tokenizer.pad_token = tokenizer.eos_token

print(f"モデル: {model_name}  デバイス: {device}")
print(f"パラメータ数: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# STEP 1: トークン化
text = "The future of AI is"
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("STEP 1: トークン化")
print(f"  入力   : {text}")
print(f"  トークン: {tokens}")
print(f"  ID     : {token_ids}")
print(f"  トークン数: {len(token_ids)}")

In [ ]:
# STEP 2: 埋め込み化
input_tensor = torch.tensor([token_ids]).to(device)
with torch.no_grad():
    embeddings = model.transformer.wte(input_tensor)

print("STEP 2: 埋め込み化")
print(f"  入力 shape  : {input_tensor.shape}  (batch=1, seq={len(token_ids)})")
print(f"  埋め込み shape: {embeddings.shape}  (batch, seq, dim=768)")
print(f"  最初のトークンの埋め込み (先頭5次元): {embeddings[0,0,:5].cpu().numpy().round(3)}")

In [ ]:
# STEP 3-4: Transformer → ロジット → 確率 (Top-5 表示)
with torch.no_grad():
    outputs = model(input_tensor)
    logits = outputs.logits  # shape: [1, seq_len, vocab_size]

last_logits = logits[0, -1, :]  # 最後トークンのロジット

for temp in [0.5, 1.0, 1.5]:
    scaled = last_logits / temp
    probs  = torch.softmax(scaled, dim=-1)
    top_probs, top_ids = torch.topk(probs, 5)
    print(f"\nSTEP 5: Top-5 候補 (temperature={temp})")
    for p, idx in zip(top_probs, top_ids):
        word = tokenizer.decode([idx.item()])
        bar = "▓" * int(p.item() * 30)
        print(f"  {p.item():.4f} [{bar:<20}] '{word}'")

## 2. ハイパーパラメータの効果比較

In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

def generate(prompt, **kwargs):
    ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            ids,
            max_new_tokens=30,
            pad_token_id=tokenizer.eos_token_id,
            **kwargs
        )
    texts = [tokenizer.decode(o, skip_special_tokens=True) for o in out]
    return texts

prompt = "Once upon a time, there was"
configs = [
    ("グリーディ (決定的)",    dict(do_sample=False)),
    ("低温度 0.5 (保守的)",    dict(do_sample=True, temperature=0.5,  top_p=0.9)),
    ("高温度 1.5 (創造的)",    dict(do_sample=True, temperature=1.5,  top_p=0.9)),
    ("Top-K=10 (絞り込み)",    dict(do_sample=True, top_k=10,         temperature=1.0)),
    ("ビームサーチ beams=4",   dict(num_beams=4,    do_sample=False, num_return_sequences=4)),
]

for name, cfg in configs:
    results = generate(prompt, **cfg)
    print(f"\n--- {name} ---")
    for i, r in enumerate(results[:2]):
        print(f"  [{i+1}] {r}")

## 3. バッチ推論と最適化

In [ ]:
import time

prompts = ["AI is", "The future of", "Machine learning can", "Python is great for"]
tokenizer.pad_token = tokenizer.eos_token

# 方法1: 逐次処理
start = time.time()
results_seq = []
for p in prompts:
    ids = tokenizer.encode(p, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=15, pad_token_id=tokenizer.eos_token_id)
    results_seq.append(tokenizer.decode(out[0], skip_special_tokens=True))
t_seq = time.time() - start

# 方法2: バッチ処理
start = time.time()
enc = tokenizer(prompts, padding=True, return_tensors="pt").to(device)
with torch.no_grad():
    outs = model.generate(**enc, max_new_tokens=15, pad_token_id=tokenizer.eos_token_id)
results_batch = [tokenizer.decode(o, skip_special_tokens=True) for o in outs]
t_batch = time.time() - start

print(f"逐次処理: {t_seq:.2f}秒")
print(f"バッチ処理: {t_batch:.2f}秒  (速度比: {t_seq/t_batch:.1f}x)")
print("\nバッチ結果:")
for p, r in zip(prompts, results_batch):
    print(f"  {p:25s} → {r[len(p):][:40]}")

## 4. トラブルシューティング

In [ ]:
# よくあるエラーパターンと安全なコード例
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

def safe_generate(model, tokenizer, prompt, max_new_tokens=30, device=None):
    """エラーに強い生成関数"""
    if device is None:
        device = next(model.parameters()).device

    # ① pad_token が未設定の場合の対処
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # ② 入力テキストが空の場合の対処
    if not prompt.strip():
        return "[空の入力]"

    # ③ max_length 超過の対処
    ids = tokenizer.encode(prompt, return_tensors="pt",
                           truncation=True, max_length=512).to(device)

    try:
        with torch.no_grad():
            out = model.generate(
                ids,
                max_new_tokens=max_new_tokens,
                pad_token_id=tokenizer.eos_token_id,
                do_sample=True, temperature=0.8, top_p=0.95,
            )
        return tokenizer.decode(out[0], skip_special_tokens=True)
    except RuntimeError as e:
        # ④ OOM 対処: CPU にフォールバック
        print(f"[Warning] GPU エラー: {e} → CPU で再試行")
        model.to("cpu"); ids = ids.to("cpu")
        with torch.no_grad():
            out = model.generate(ids, max_new_tokens=max_new_tokens)
        return tokenizer.decode(out[0], skip_special_tokens=True)

tok = GPT2Tokenizer.from_pretrained("gpt2")
mdl = GPT2LMHeadModel.from_pretrained("gpt2")
result = safe_generate(mdl, tok, "The history of computing is")
print("生成結果:", result)

## ✅ 理解度チェック

- [ ] トークン化・埋め込み・Transformer の 3 ステップが説明できる  
- [ ] temperature が高いほどランダムになることを確認した  
- [ ] バッチ処理が逐次処理より高速な理由が分かった  

---
✅ 完了 → **[段階3: 実践応用編](05_advanced_implementation_guide.ipynb)** へ